# Imports

In [1]:
import torch
from transformers import BitsAndBytesConfig
from tokenizers import SentencePieceBPETokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import PreTrainedTokenizerFast, T5Tokenizer
from peft import LoraConfig, get_peft_model
import sentencepiece as spm

/home/vlad/github/volynvlad/mova-be-llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

In [3]:
load_dotenv("../.env")

True

# Tokenizer

In [4]:
MODEL_FILE = "../models/tokenizer/bel_tokenizer.model"

In [5]:
MODEL_NAME = "google/gemma-3-1b-it"

In [6]:
sp = spm.SentencePieceProcessor(model_file=MODEL_FILE)
print(sp.encode("Hello world!", out_type=int))

[738, 2023, 31914, 24022, 31955]


In [7]:
# tokenizer = SentencePieceBPETokenizer("../models/tokenizer/bel_tokenizer.vocab", MODEL_FILE)
# tokenizer.save("../models/tokenizer/hf_bel_tokenizer.json")

In [8]:
tokenizer = T5Tokenizer(
    vocab_file=MODEL_FILE,
    # tokenizer_object=sp,
    pad_token="[PAD]",
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    bos_token="<s>",
    eos_token="</s>"
)
tokenizer.save_pretrained("../models/tokenizer/hf_bel_tokenizer")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


('../models/tokenizer/hf_bel_tokenizer/tokenizer_config.json',
 '../models/tokenizer/hf_bel_tokenizer/special_tokens_map.json',
 '../models/tokenizer/hf_bel_tokenizer/spiece.model',
 '../models/tokenizer/hf_bel_tokenizer/added_tokens.json')

In [21]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=os.getenv("HF_TOKEN"))

inputs = tokenizer("Вітам!", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)

In [24]:
tokenizer

T5Tokenizer(name_or_path='', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extr

In [19]:
outputs[0]

tensor([   100,     14,    485,   7668,  19391,  31955,      2,  19174,   1688,
          3117,    718,   3001, 236787,   1192,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,   9503,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  10671,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  38852,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  28975,    905,    107,  19174,   1688,
          3117,    718,   3001])

In [20]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

IndexError: piece id is out of range.

# Config

In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Fine-tuning

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)